# Airbnb summary listings Bronze ingestion (F07.2-T01)

Ingests the Toronto Inside Airbnb `listings.csv` summary file into its
own Bronze data product (`airbnb_listings_summary_bronze`) - separate
from the detailed listings, calendar and reviews products, and never
unioned with detailed listing facts.

The complete ingestion flow lives ONCE in `src/airbnb/bronze_flow.py`
and is shared by every per-dataset Bronze notebook: manifest-record
resolution by explicit `source_file_id` (seven registered files share
the family/entity/version identity `insideairbnb/toronto/2026-09-16`,
so resolving by identity alone is ambiguous), immutable-landing
verification (hash + size, plain-CSV header fingerprint, contract-
required columns), string-preserving reads, contract-driven
row-quality retention (invalid rows are RETAINED with
`_quality_status` / `_quality_reason`, never dropped), the
unified ingestion manifest (F07.4) and the atomic `replaceWhere`
slice write with reconciliation asserts.

This notebook supplies only the summary listings dataset spec, the run
widgets and the result display. City, snapshot date and source family
are derived from the landed manifest record itself - never hard-coded.

In [0]:
# Run-mode controls (F02.2 / F05.3 conventions, applied to the summary
# listings dataset in F07.2-T01).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved into the dedicated trippilot_test catalog through
# src.common.environment. "live" resolves the development tables and is
# used only after the test-mode evidence gate has passed.
#
# force_reload is the explicit reprocess authorization: it permits an
# exact rerun of an already-ingested file and the controlled replacement
# of a corrected source file (a different source_file_id).
#
# source_file_id pins the EXACT landed manifest record this run reads:
# seven files share the family/entity/version identity
# insideairbnb/toronto/2026-09-16 (detailed listings/calendar/reviews
# csv.gz + summary listings/reviews csv + neighbourhoods csv/geojson),
# so resolving by identity alone is ambiguous - the explicit id is
# required.
dbutils.widgets.dropdown("run_mode", "test", ["test", "live"])
dbutils.widgets.text("force_reload", "false")
dbutils.widgets.text(
    "source_file_id",
    "96ec34fbd40086336cace9d6d5eae9aea0dbedbfdd2d4153feee0538ec4d7032",
)

RUN_MODE = dbutils.widgets.get("run_mode")
FORCE_RELOAD = dbutils.widgets.get("force_reload")
SOURCE_FILE_ID = dbutils.widgets.get("source_file_id")

print(f"Run mode: {RUN_MODE} | force_reload: {FORCE_RELOAD}")
print(f"source_file_id: {SOURCE_FILE_ID}")

In [0]:
# Locate the repository root, then run the shared flow (F07.2-T01).
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the TripPilot repository root (src/ and config/)."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.airbnb.bronze_flow import BronzeDatasetSpec, run_bronze_ingestion

# Dataset identity for the summary listings path (F07.1 evidence): the
# shared flow derives city/snapshot/source from the landed manifest record.
LISTINGS_SUMMARY_SPEC = BronzeDatasetSpec(
    dataset="listings_summary",
    schema_label="summary-listings",
    contract_path="contracts/airbnb/bronze_listings_summary.yml",
    rules_key="bronze_listings_summary",
    bronze_key="listings_summary_bronze",
    default_source_file_id="96ec34fbd40086336cace9d6d5eae9aea0dbedbfdd2d4153feee0538ec4d7032",
)

summary = run_bronze_ingestion(
    LISTINGS_SUMMARY_SPEC,
    spark,
    source_file_id=SOURCE_FILE_ID,
    run_mode=RUN_MODE,
    force_reload=FORCE_RELOAD,
)

In [0]:
# Result display: F07.2 acceptance evidence (raw sample, retained
# invalid rows, per-identity verification, per-slice summary).
display(summary["raw_df"].limit(5))

if summary["invalid_rows"]:
    from pyspark.sql import functions as F

    display(
        summary["bronze_df"].filter(F.col("_quality_status") == "invalid").limit(20)
    )

display(summary["verification_df"])
display(summary["snapshot_summary"])